# Test Usage Logs - AskMe

**Notebook simple pour tester la route `/api/usage/logs`**

- Route sans authentification
- Créé automatiquement le container s'il n'existe pas
- Récupère tous les logs d'usage token

In [ ]:
import requests
import pandas as pd
from datetime import datetime

print(f"🔍 Test Usage Logs API - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

try:
    # Appel simple sans authentification
    response = requests.get('http://localhost:50505/api/usage/logs')
    
    print(f"Status: {response.status_code}")
    
    if response.status_code == 200:
        data = response.json()
        
        if data.get('success'):
            records = data.get('records', [])
            total = data.get('total_records', 0)
            
            print(f"📊 Total enregistrements: {total}")
            
            if records:
                # DataFrame pour affichage
                df_data = []
                for record in records:
                    df_data.append({
                        'Timestamp': record.get('timestamp', '')[:19] if record.get('timestamp') else 'N/A',
                        'User': record.get('user_id', 'N/A')[:15],
                        'Provider': record.get('provider', 'N/A'),
                        'Model': record.get('model', 'N/A')[:20] if record.get('model') else 'N/A',
                        'Input': record.get('input_tokens', 0),
                        'Output': record.get('output_tokens', 0),
                        'Total': record.get('total_tokens', 0),
                        'Conv_ID': record.get('conversation_id', 'N/A')[:10]
                    })
                
                df = pd.DataFrame(df_data)
                
                print("\n📋 LOGS D'USAGE (derniers 10):")
                print(df.head(10).to_string(index=False))
                
                # Stats par provider
                if len(df) > 0:
                    print("\n📈 STATISTIQUES PAR PROVIDER:")
                    stats = df.groupby('Provider').agg({
                        'Total': ['sum', 'count', 'mean']
                    }).round(1)
                    stats.columns = ['Total Tokens', 'Nb Requêtes', 'Tokens Moyens']
                    print(stats)
                    
                    print(f"\n🎯 RÉSUMÉ:")
                    print(f"   • Total tokens système: {df['Total'].sum():,}")
                    print(f"   • Providers actifs: {len(df['Provider'].unique())}")
                    print(f"   • Utilisateurs: {len(df['User'].unique())}")
                    print(f"   • Dernière activité: {df['Timestamp'].iloc[0]}")
            else:
                print("\n⚠️ Aucun log d'usage trouvé")
                print("💡 Container créé automatiquement - envoyez quelques questions via AskMe")
        else:
            print(f"❌ Erreur API: {data.get('error', 'Unknown error')}")
    
    elif response.status_code == 503:
        print("⚠️ Service indisponible - Usage tracking désactivé")
    elif response.status_code == 404:
        print("❌ Route non trouvée")
    else:
        try:
            error_data = response.json()
            print(f"❌ Erreur HTTP {response.status_code}: {error_data.get('error', response.text[:200])}")
        except:
            print(f"❌ Erreur HTTP {response.status_code}: {response.text[:200]}")
        
except requests.exceptions.ConnectionError:
    print("❌ Connexion refusée - AskMe n'est pas démarré")
    print("💡 Lancez: python -m uvicorn app:app --port 50505 --reload")
except Exception as e:
    print(f"❌ Exception: {e}")

print("\n" + "="*60)
print("✨ Test terminé!")

## 💡 Instructions

1. **Assurez-vous qu'AskMe tourne** sur `localhost:50505`
2. **Lancez la cellule ci-dessus** pour voir les logs d'usage
3. **Container créé automatiquement** si il n'existe pas
4. **Aucune authentification** requise pour cette route de test

### Route utilisée:
- `GET /api/usage/logs` - Récupère tous les logs (sans auth)

### Fonctionnalités:
- ✅ Création automatique du container
- ✅ Affichage des statistiques
- ✅ Format simple et lisible